In [1]:
# ============================================
# Recommendation System using Cosine Similarity
# Anime Recommendation Project
# ============================================

# Import Libraries
import pandas as pd
import numpy as np

from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import MinMaxScaler


In [2]:
# --------------------------------------------
# 1. Load Dataset
# --------------------------------------------

df = pd.read_csv("anime.csv")

# Display first 5 rows
print(df.head())

# Dataset Information
print("\nDataset Info:")
print(df.info())


   anime_id                              name  \
0     32281                    Kimi no Na wa.   
1      5114  Fullmetal Alchemist: Brotherhood   
2     28977                          Gintama°   
3      9253                       Steins;Gate   
4      9969                     Gintama&#039;   

                                               genre   type episodes  rating  \
0               Drama, Romance, School, Supernatural  Movie        1    9.37   
1  Action, Adventure, Drama, Fantasy, Magic, Mili...     TV       64    9.26   
2  Action, Comedy, Historical, Parody, Samurai, S...     TV       51    9.25   
3                                   Sci-Fi, Thriller     TV       24    9.17   
4  Action, Comedy, Historical, Parody, Samurai, S...     TV       51    9.16   

   members  
0   200630  
1   793665  
2   114262  
3   673572  
4   151266  

Dataset Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12294 entries, 0 to 12293
Data columns (total 7 columns):
 #   Column    Non-Null

In [3]:
# --------------------------------------------
# 2. Data Preprocessing
# --------------------------------------------

# Check missing values
print("\nMissing Values:")
print(df.isnull().sum())

# Fill missing values

# Genre missing values
df['genre'] = df['genre'].fillna('Unknown')

# Type missing values
df['type'] = df['type'].fillna('Unknown')

# Rating missing values
df['rating'] = df['rating'].fillna(df['rating'].mean())

# Episodes missing values
df['episodes'] = df['episodes'].replace('Unknown', np.nan)
df['episodes'] = pd.to_numeric(df['episodes'], errors='coerce')
df['episodes'] = df['episodes'].fillna(df['episodes'].median())

# Members missing values
df['members'] = df['members'].fillna(df['members'].median())



Missing Values:
anime_id      0
name          0
genre        62
type         25
episodes      0
rating      230
members       0
dtype: int64


In [4]:
# --------------------------------------------
# 3. Exploratory Data Analysis
# --------------------------------------------

print("\nDataset Shape:", df.shape)

print("\nColumns:")
print(df.columns)

print("\nStatistical Summary:")
print(df.describe())


Dataset Shape: (12294, 7)

Columns:
Index(['anime_id', 'name', 'genre', 'type', 'episodes', 'rating', 'members'], dtype='object')

Statistical Summary:
           anime_id      episodes        rating       members
count  12294.000000  12294.000000  12294.000000  1.229400e+04
mean   14058.221653     12.095412      6.473902  1.807134e+04
std    11455.294701     46.244062      1.017096  5.482068e+04
min        1.000000      1.000000      1.670000  5.000000e+00
25%     3484.250000      1.000000      5.900000  2.250000e+02
50%    10260.500000      2.000000      6.550000  1.550000e+03
75%    24794.500000     12.000000      7.170000  9.437000e+03
max    34527.000000   1818.000000     10.000000  1.013917e+06


In [5]:
# --------------------------------------------
# 4. Feature Extraction
# --------------------------------------------

# We will use:
# Genre + Type + Rating + Episodes + Members

# ----- TEXT FEATURES -----

# Convert genres into text vectors
cv = CountVectorizer(stop_words='english')

genre_matrix = cv.fit_transform(df['genre'])

# ----- NUMERICAL FEATURES -----

numerical_features = df[['rating', 'episodes', 'members']]

# Normalize numerical features
scaler = MinMaxScaler()

numerical_scaled = scaler.fit_transform(numerical_features)

# ----- COMBINE FEATURES -----

# Convert sparse matrix to array
genre_array = genre_matrix.toarray()

# Combine text and numerical features
combined_features = np.hstack((genre_array, numerical_scaled))

print("\nCombined Feature Shape:", combined_features.shape)


Combined Feature Shape: (12294, 50)


In [6]:
# --------------------------------------------
# 5. Compute Cosine Similarity
# --------------------------------------------

cosine_sim = cosine_similarity(combined_features)

print("\nCosine Similarity Matrix Shape:")
print(cosine_sim.shape)



Cosine Similarity Matrix Shape:
(12294, 12294)


In [7]:
# --------------------------------------------
# 6. Recommendation Function
# --------------------------------------------

def recommend_anime(anime_name, similarity_threshold=0.5, top_n=10):
    
    # Check whether anime exists
    if anime_name not in df['name'].values:
        return "Anime not found in dataset."
    
    # Get index of anime
    idx = df[df['name'] == anime_name].index[0]
    
    # Similarity scores
    sim_scores = list(enumerate(cosine_sim[idx]))
    
    # Sort by similarity score
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
    
    # Remove the anime itself
    sim_scores = sim_scores[1:]

    # Apply threshold
    filtered_scores = [x for x in sim_scores if x[1] >= similarity_threshold]
    
    # Top N recommendations
    filtered_scores = filtered_scores[:top_n]
    
    # Create recommendation list
    recommendations = []
    
    for i, score in filtered_scores:
        recommendations.append({
            'Anime': df.iloc[i]['name'],
            'Genre': df.iloc[i]['genre'],
            'Rating': df.iloc[i]['rating'],
            'Similarity Score': round(score, 3)
        })
    
    return pd.DataFrame(recommendations)


In [8]:
# --------------------------------------------
# 7. Example Recommendations
# --------------------------------------------

anime_title = "Naruto"

recommendations = recommend_anime(
    anime_name=anime_title,
    similarity_threshold=0.6,
    top_n=10
)

print(f"\nRecommendations for {anime_title}:")
print(recommendations)



Recommendations for Naruto:
                                               Anime  \
0                                 Naruto: Shippuuden   
1        Naruto: Shippuuden Movie 4 - The Lost Tower   
2  Naruto: Shippuuden Movie 3 - Hi no Ishi wo Tsu...   
3                           Boruto: Naruto the Movie   
4                                        Naruto x UT   
5  Naruto Soyokazeden Movie: Naruto to Mashin to ...   
6  Boruto: Naruto the Movie - Naruto ga Hokage ni...   
7               Naruto Shippuuden: Sunny Side Battle   
8                            Kyutai Panic Adventure!   
9                                      Dragon Ball Z   

                                               Genre  Rating  Similarity Score  
0  Action, Comedy, Martial Arts, Shounen, Super P...    7.94             0.998  
1  Action, Comedy, Martial Arts, Shounen, Super P...    7.53             0.977  
2  Action, Comedy, Martial Arts, Shounen, Super P...    7.50             0.977  
3  Action, Comedy, Martial Art

In [9]:
# --------------------------------------------
# 8. Experiment with Thresholds
# --------------------------------------------

thresholds = [0.3, 0.5, 0.7]

for t in thresholds:
    print(f"\nThreshold = {t}")
    
    recs = recommend_anime(
        anime_name="Naruto",
        similarity_threshold=t,
        top_n=5
    )
    
    print(recs)


Threshold = 0.3
                                               Anime  \
0                                 Naruto: Shippuuden   
1        Naruto: Shippuuden Movie 4 - The Lost Tower   
2  Naruto: Shippuuden Movie 3 - Hi no Ishi wo Tsu...   
3                           Boruto: Naruto the Movie   
4                                        Naruto x UT   

                                               Genre  Rating  Similarity Score  
0  Action, Comedy, Martial Arts, Shounen, Super P...    7.94             0.998  
1  Action, Comedy, Martial Arts, Shounen, Super P...    7.53             0.977  
2  Action, Comedy, Martial Arts, Shounen, Super P...    7.50             0.977  
3  Action, Comedy, Martial Arts, Shounen, Super P...    8.03             0.976  
4  Action, Comedy, Martial Arts, Shounen, Super P...    7.58             0.972  

Threshold = 0.5
                                               Anime  \
0                                 Naruto: Shippuuden   
1        Naruto: Shippuuden Mov

In [10]:
# --------------------------------------------
# 9. Observations / Analysis
# --------------------------------------------

print("""
Analysis:
1. Lower threshold values provide more recommendations.
2. Higher threshold values provide fewer but more similar recommendations.
3. Genre heavily influences similarity.
4. Numerical features like rating and members improve recommendation quality.

Areas of Improvement:
- Use TF-IDF instead of CountVectorizer.
- Apply collaborative filtering techniques.
- Use user-item interaction matrix.
- Implement hybrid recommendation systems.
- Use deep learning models for advanced recommendations.
""")


Analysis:
1. Lower threshold values provide more recommendations.
2. Higher threshold values provide fewer but more similar recommendations.
3. Genre heavily influences similarity.
4. Numerical features like rating and members improve recommendation quality.

Areas of Improvement:
- Use TF-IDF instead of CountVectorizer.
- Apply collaborative filtering techniques.
- Use user-item interaction matrix.
- Implement hybrid recommendation systems.
- Use deep learning models for advanced recommendations.

